In [6]:
import os
import json

training_data_dir = ".\\training_data\\"
all_FSP_data = []

for filename in os.listdir(training_data_dir):
    # Check if the file is a JSON file before trying to open it
    if filename.endswith('.json'):
        full_filename = os.path.join(training_data_dir, filename)
        
        try:
            with open(full_filename, 'r', encoding='utf-8') as file:
                data = json.load(file)
            all_FSP_data.append(data)
        except json.JSONDecodeError:
            # This prevents a malformed JSON file from crashing the script
            print(f"Warning: Could not decode JSON from file: {filename}")

In [10]:
def get_explicit_content(entry):
    explicit_ids = entry['final_censored_ids']
    transcript = entry['transcript']

    for line in transcript:
        text = line.get('line_text', [])
        if not text:
            continue

        explicit_phrases = set()

        for d in line['line_words']:
            if d['id'] in explicit_ids:
                explicit_phrases.add(d['text'])

        is_explicit = True if explicit_phrases else False
        # print("text:", text)
        # print("explicit phrases:", list(explicit_phrases))     
        # print("is explicit:", is_explicit)
        # print()

        d = {
            "sentence": text,
            "explicit_phrases": list(explicit_phrases),
            "is_explicit": is_explicit
        }

        data.append(d)

data = []

for entry in all_FSP_data:
    get_explicit_content(entry)

output_filename = 'formatted_WMSE_data.jsonl'

with open(output_filename, 'w', encoding='utf-8') as file:
    for item in data:
        # Convert the dictionary to a JSON string
        json_string = json.dumps(item)
        # Write the JSON string to the file, followed by a newline
        file.write(json_string + '\n')